# 01 — Análise Exploratória (EDA)

**Objetivo:** entender a estrutura dos dados antes de qualquer modelagem.

Roteiro:
1. Carregar a base sintética semanal  
2. Inspeção geral (shape, dtypes, missing values)  
3. Distribuição estatística das variáveis  
4. Evolução temporal da variável dependente (novos assinantes)  
5. Evolução do spend por canal  
6. Correlação entre spend e assinantes  
7. Sazonalidade e tendência

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Estilo visual
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110

print("Bibliotecas carregadas com sucesso.")

## 1. Carregar dados

Se a base ainda não foi gerada, rode primeiro:
```bash
python data/generate_synthetic_data.py
```

In [ ]:
DATA_PATH = "../data/MMM_Synth_Weekly_Data.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["week"])
df = df.sort_values("week").reset_index(drop=True)

# Identificar colunas de spend
spend_cols   = [c for c in df.columns if c.startswith("spend_")]
contrib_cols = [c for c in df.columns if c.startswith("contrib_")]

# Nomes legíveis dos canais
CHANNEL_NAMES = {
    "paid_search_google_":     "Paid Search (Google)",
    "social_media_meta_ads_":  "Social Media (Meta Ads)",
    "social_media_youtube_ads_": "Social Media (YouTube Ads)",
    "affiliate_marketing":     "Affiliate Marketing",
    "email_marketing":         "Email Marketing",
    "e-commerce_company_site": "E-commerce Company Site",
    "call_center":             "Call Center",
}

print(f"Shape: {df.shape}")
print(f"Período: {df['week'].min().date()} → {df['week'].max().date()}")
print(f"\nColunas de spend ({len(spend_cols)}): {spend_cols}")
df.head(3)

## 2. Inspeção geral

In [ ]:
# Tipos e valores missing
info = pd.DataFrame({
    "dtype":   df.dtypes,
    "missing": df.isnull().sum(),
    "missing_%": (df.isnull().mean() * 100).round(2),
})
print(info.to_string())

In [ ]:
# Estatísticas descritivas das colunas numéricas
df[spend_cols + ["new_subscribers"]].describe().applymap(lambda x: f"{x:,.0f}")

## 3. Evolução temporal — Novos Assinantes

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(df["week"], df["new_subscribers"], color="#0f3460", lw=1.5)
ax.fill_between(df["week"], df["new_subscribers"], alpha=0.15, color="#0f3460")

# Divisor treino / holdout (últimas 24 semanas)
holdout_start = df["week"].iloc[-24]
ax.axvline(holdout_start, color="#e94560", lw=1.5, linestyle="--", label="Início holdout")
ax.text(holdout_start, ax.get_ylim()[1] * 0.97, " Holdout", color="#e94560", fontsize=9)

ax.set_title("Novos Assinantes Semanais — Video+ (Jan/2023–Dez/2024)", fontweight="bold")
ax.set_ylabel("Novos Assinantes")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/eda_subscribers_over_time.png", dpi=130, bbox_inches="tight")
plt.show()
print("Média semanal:", f"{df['new_subscribers'].mean():,.0f}")
print("Total 2023:",    f"{df[df['week'].dt.year == 2023]['new_subscribers'].sum():,}")
print("Total 2024:",    f"{df[df['week'].dt.year == 2024]['new_subscribers'].sum():,}")

## 4. Evolução do Spend por Canal

In [ ]:
fig, axes = plt.subplots(len(spend_cols), 1, figsize=(14, 3 * len(spend_cols)), sharex=True)

colors = sns.color_palette("tab10", len(spend_cols))

for ax, col, color in zip(axes, spend_cols, colors):
    label = col.replace("spend_", "").replace("_", " ").title()
    ax.plot(df["week"], df[col] / 1000, color=color, lw=1.3)
    ax.fill_between(df["week"], df[col] / 1000, alpha=0.12, color=color)
    ax.set_ylabel("R$ mil", fontsize=9)
    ax.set_title(label, fontsize=10, fontweight="bold", loc="left")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}k"))
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Semana")
fig.suptitle("Spend Semanal por Canal (R$)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../outputs/eda_spend_by_channel.png", dpi=130, bbox_inches="tight")
plt.show()

## 5. Spend Total por Canal — Participação

In [ ]:
spend_total = df[spend_cols].sum().sort_values(ascending=False)
labels      = [c.replace("spend_", "").replace("_", " ").title() for c in spend_total.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras
axes[0].barh(labels[::-1], spend_total.values[::-1] / 1e6,
             color=sns.color_palette("tab10", len(labels)))
axes[0].set_xlabel("Spend Total (R$ M)")
axes[0].set_title("Spend Total por Canal (2 anos)", fontweight="bold")

# Pizza
axes[1].pie(spend_total.values, labels=labels, autopct="%1.1f%%",
            colors=sns.color_palette("tab10", len(labels)), startangle=90)
axes[1].set_title("Participação no Budget", fontweight="bold")

plt.tight_layout()
plt.show()

print("\nSpend total por canal (R$ M):")
for col, total in spend_total.items():
    print(f"  {col.replace('spend_', ''):40s} R$ {total/1e6:.2f} M")
print(f"  {'TOTAL':40s} R$ {spend_total.sum()/1e6:.2f} M")

## 6. Correlação entre Spend e Novos Assinantes

> ⚠️ Correlação linear simples ignora adstock e saturação — é apenas exploratória.

In [ ]:
corr_data = df[spend_cols + ["new_subscribers"]].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_data, dtype=bool))
sns.heatmap(corr_data, mask=mask, annot=True, fmt=".2f",
            cmap="RdYlGn", center=0, ax=ax,
            xticklabels=[c.replace("spend_", "") for c in corr_data.columns],
            yticklabels=[c.replace("spend_", "") for c in corr_data.columns])
ax.set_title("Correlação de Pearson — Spend × Assinantes", fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Sazonalidade e Tendência

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

# Assinantes
axes[0].plot(df["week"], df["new_subscribers"], color="#0f3460", lw=1.3)
axes[0].set_ylabel("Novos Assinantes")
axes[0].set_title("Decomposição: Assinantes | Sazonalidade | Tendência", fontweight="bold")

# Índice de sazonalidade
axes[1].plot(df["week"], df["seasonality_index"], color="#e94560", lw=1.3)
axes[1].axhline(1.0, color="gray", lw=0.8, linestyle="--")
axes[1].set_ylabel("Índice Sazonal")

# Tendência
axes[2].plot(df["week"], df["trend"], color="#16213e", lw=1.3)
axes[2].set_ylabel("Tendência")
axes[2].set_xlabel("Semana")

for ax in axes:
    ax.grid(True, alpha=0.3)
    # Marcar flag de reajuste
    for _, row in df[df["price_hike_flag"] == 1].iterrows():
        ax.axvline(row["week"], color="orange", lw=1.0, linestyle=":", alpha=0.8)

plt.tight_layout()
plt.savefig("../outputs/eda_seasonality_trend.png", dpi=130, bbox_inches="tight")
plt.show()
print("Semanas com reajuste de preço:", df[df["price_hike_flag"] == 1]["week"].dt.date.tolist())

## Conclusões da EDA

- **Tendência de crescimento** clara na série de assinantes ao longo dos 2 anos.
- **Sazonalidade anual visível**: picos no início do ano e na segunda metade.
- **Flags de reajuste de preço** (jul/2023 e jul/2024) causam impacto negativo pontual.
- **Multicolinearidade entre canais**: spend correlacionado — requer diagnóstico VIF no modelo.
- Correlação linear simples subestima a relação real com os assinantes (efeito de adstock e saturação).

➡ Próximo passo: **notebook 02** — pipeline de adstock + saturação de Hill.